In [7]:
import pandas as pd

df = pd.read_parquet("../data/processed/orders_clean.parquet")
print(df.head())   



  order_id user_id  amount  quantity            created_at    status  \
0    A4434    0021  479.46         6  2025-10-06T15:00:00Z  refunded   
1    A0177    0677  124.64         4  2025-06-15T01:35:00Z  Refunded   
2    A2972    0763    <NA>         4  2025-11-14T23:05:00Z  Refunded   
3    A2288    0197  397.26         7  2025-10-16T20:04:00Z      Paid   
4    A3985    0460  374.25         1  2025-01-16T03:21:00Z   Pending   

  status_clean  amount__isna  quantity__isna  
0       refund         False           False  
1       refund         False           False  
2       refund          True           False  
3         paid         False           False  
4      pending         False           False  


In [3]:
import pandas as pd

df = pd.read_parquet("../data/processed/analytics_table.parquet")
print(df.head())   



  order_id user_id  amount  quantity                created_at    status  \
0    A4630    0285  446.67         9 2025-01-01 05:46:00+00:00      Paid   
1    A4103    0794   87.37         2 2025-01-01 09:07:00+00:00      paid   
2    A1280    0814  498.89         8 2025-01-01 11:06:00+00:00  refunded   
3    A3546    0553  254.52         5 2025-01-01 11:34:00+00:00   Pending   
4    A2880    0536  112.01         1 2025-01-01 11:39:00+00:00  refunded   

  status_clean  amount__isna  quantity__isna        date    year    month  \
0         paid         False           False  2025-01-01  2025.0  2025-01   
1         paid         False           False  2025-01-01  2025.0  2025-01   
2       refund         False           False  2025-01-01  2025.0  2025-01   
3      pending         False           False  2025-01-01  2025.0  2025-01   
4       refund         False           False  2025-01-01  2025.0  2025-01   

         dow  hour country signup_date  amount_winsor  amount__is_outlier  
0  W

In [9]:
import pandas as pd
import re

_ws = re.compile(r"\s+")


In [10]:
def missingness_report(df):
    return df.isna().sum().rename("n_missing").to_frame().assign(p_missing=lambda t: t["n_missing"]/len(df)).sort_values("p_missing", ascending=False)

def add_missing_flags(df, cols):
    out = df.copy()
    for c in cols:
        out[f"{c}__isna"] = out[c].isna()
    return out

def normalize_text(s):
    return s.astype("string").str.lower().str.replace(_ws," ",regex=True).str.strip().str.casefold()

def apply_mapping(s, mapping):
    return s.map(lambda x: mapping.get(x,x))

def remove_duplicates(df, key_cols, ts_col):
    return df.sort_values(ts_col).drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

def enforce_schema_orders(df):
    return df.assign(
        order_id=df["order_id"].astype("string"),
        user_id=df["user_id"].astype("string"),
        amount=pd.to_numeric(df["amount"], errors="coerce").astype("Float64"),
        quantity=pd.to_numeric(df["quantity"], errors="coerce").astype("Int64")
    )

def enforce_schema_users(df):
    return df.assign(
        user_id=df["user_id"].astype("string")
    )

In [12]:
orders = pd.read_csv("../data/raw/orders_small.csv")
users = pd.read_csv("../data/raw/users_small.csv")

In [13]:
orders = remove_duplicates(orders, ["order_id"], "created_at")
users = remove_duplicates(users, ["user_id"], "signup_date")


In [14]:
orders = enforce_schema_orders(orders)
users = enforce_schema_users(users)


In [15]:
rep_orders = missingness_report(orders)
rep_users = missingness_report(users)

print("Missingness Orders:\n", rep_orders)
print("\nMissingness Users:\n", rep_users)


Missingness Orders:
             n_missing  p_missing
amount              1        0.2
order_id            0        0.0
user_id             0        0.0
quantity            0        0.0
created_at          0        0.0
status              0        0.0

Missingness Users:
              n_missing  p_missing
user_id              0        0.0
country              0        0.0
signup_date          0        0.0


In [16]:
status_norm = normalize_text(orders["status"])
status_clean = apply_mapping(status_norm, {"paid":"paid","refund":"refund","refunded":"refund"})
orders["status_clean"] = status_clean

orders = add_missing_flags(orders, ["amount","quantity"])
orders.head()


,order_id,user_id,amount,quantity,created_at,status,status_clean,amount__isna,quantity__isna
0,A3985,460,374.25,1,2025-01-16T03:21:00Z,Pending,pending,False,False
1,A0177,677,124.64,4,2025-06-15T01:35:00Z,Refunded,refund,False,False
2,A4434,21,479.46,6,2025-10-06T15:00:00Z,refunded,refund,False,False
3,A2288,197,397.26,7,2025-10-16T20:04:00Z,Paid,paid,False,False
4,A2972,763,<NA>,4,2025-11-14T23:05:00Z,Refunded,refund,True,False


In [17]:
orders["user_id"] = orders["user_id"].astype("string")
users["user_id"] = users["user_id"].astype("string")


In [18]:
joined = orders.merge(users, on="user_id", how="left", suffixes=("", "_user"))
joined.head()


,order_id,user_id,amount,quantity,created_at,status,status_clean,amount__isna,quantity__isna,country,signup_date
0,A3985,460,374.25,1,2025-01-16T03:21:00Z,Pending,pending,False,False,NaN,NaN
1,A0177,677,124.64,4,2025-06-15T01:35:00Z,Refunded,refund,False,False,NaN,NaN
2,A4434,21,479.46,6,2025-10-06T15:00:00Z,refunded,refund,False,False,NaN,NaN
3,A2288,197,397.26,7,2025-10-16T20:04:00Z,Paid,paid,False,False,NaN,NaN
4,A2972,763,<NA>,4,2025-11-14T23:05:00Z,Refunded,refund,True,False,NaN,NaN


In [20]:
print("Orders before join:")
print(orders.head())
print("\nUsers before join:")
print(users.head())


Orders before join:
  order_id user_id  amount  quantity            created_at    status  \
0    A3985     460  374.25         1  2025-01-16T03:21:00Z   Pending   
1    A0177     677  124.64         4  2025-06-15T01:35:00Z  Refunded   
2    A4434      21  479.46         6  2025-10-06T15:00:00Z  refunded   
3    A2288     197  397.26         7  2025-10-16T20:04:00Z      Paid   
4    A2972     763    <NA>         4  2025-11-14T23:05:00Z  Refunded   

  status_clean  amount__isna  quantity__isna  
0      pending         False           False  
1       refund         False           False  
2       refund         False           False  
3         paid         False           False  
4       refund          True           False  

Users before join:
  user_id country signup_date
0       1      AE  2025-01-01
1       2      QA  2025-01-01
2       3      AE  2025-01-01
3       4      AE  2025-01-02
4       5      KW  2025-01-02


In [21]:
joined = orders.merge(users, on="user_id", how="left", suffixes=("", "_user"))
print("Joined table:")
print(joined.head())


Joined table:
  order_id user_id  amount  quantity            created_at    status  \
0    A3985     460  374.25         1  2025-01-16T03:21:00Z   Pending   
1    A0177     677  124.64         4  2025-06-15T01:35:00Z  Refunded   
2    A4434      21  479.46         6  2025-10-06T15:00:00Z  refunded   
3    A2288     197  397.26         7  2025-10-16T20:04:00Z      Paid   
4    A2972     763    <NA>         4  2025-11-14T23:05:00Z  Refunded   

  status_clean  amount__isna  quantity__isna country signup_date  
0      pending         False           False     NaN         NaN  
1       refund         False           False     NaN         NaN  
2       refund         False           False     NaN         NaN  
3         paid         False           False     NaN         NaN  
4       refund          True           False     NaN         NaN  


In [23]:
import pandas as pd

df = pd.read_parquet("../data/processed/analytics_table.parquet")
print(df.head())   



  order_id user_id  amount  quantity                created_at    status  \
0    A4630    0285  446.67         9 2025-01-01 05:46:00+00:00      Paid   
1    A4103    0794   87.37         2 2025-01-01 09:07:00+00:00      paid   
2    A1280    0814  498.89         8 2025-01-01 11:06:00+00:00  refunded   
3    A3546    0553  254.52         5 2025-01-01 11:34:00+00:00   Pending   
4    A2880    0536  112.01         1 2025-01-01 11:39:00+00:00  refunded   

  status_clean  amount__isna  quantity__isna        date    year    month  \
0         paid         False           False  2025-01-01  2025.0  2025-01   
1         paid         False           False  2025-01-01  2025.0  2025-01   
2       refund         False           False  2025-01-01  2025.0  2025-01   
3      pending         False           False  2025-01-01  2025.0  2025-01   
4       refund         False           False  2025-01-01  2025.0  2025-01   

         dow  hour country signup_date  amount_winsor  amount__is_outlier  
0  W